# Baseline Monte Carlo Runner

Runs (or re-runs) the baseline simulation and writes the result to `src/mc_baseline_data/`.  
The capacity scenario notebooks load this file automatically — rerun them after this completes.

| Setting | Value |
|---|---|
| Seeds | `N_SEEDS` (default 100) |
| Seed start | 42 |
| Workers | `N_WORKERS` (default 3) |
| **`FORCE_RERUN`** | **Set `True` to delete existing CSV and rerun from scratch** |

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
for _sub in ['src', 'ModelParameters']:
    _p = str(PROJECT_ROOT / _sub)
    if _p not in sys.path:
        sys.path.insert(0, _p)

import parameters as cfg
from mc_baseline import run_mc_baseline

print(f'Project root    : {PROJECT_ROOT}')
print(f'DAILY_PATIENTS  : {cfg.DAILY_PATIENTS}')
print(f'CAPACITIES      : {cfg.CAPACITIES}')
print(f'HPV_COLPO_PROB  : {cfg.HPV_POSITIVE_COLPOSCOPY_PROB}')
print(f'WARMUP_YEARS    : {cfg.WARMUP_YEARS}')

Project root    : /Users/Sophia/NYP
DAILY_PATIENTS  : 2
CAPACITIES      : {'cytology': 4, 'hpv_alone': 4, 'co_test': 4, 'ldct': 4, 'colposcopy': 4, 'lung_biopsy': 4, 'leep': 4, 'cone_biopsy': 4}
HPV_COLPO_PROB  : 0.3
WARMUP_YEARS    : 10


In [3]:
# ── settings ──────────────────────────────────────────────────────────────
N_SEEDS      = 100
N_WORKERS    = 3
SEED_START   = 42
FORCE_RERUN  = True   # True  → delete existing CSV and rerun from scratch
                       # False → skip if CSV already exists

DATA_DIR = PROJECT_ROOT / 'src' / 'mc_baseline_data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV = DATA_DIR / f'mc_baseline_n{N_SEEDS}_start{SEED_START}.csv'
print(f'Output path : {OUT_CSV}')
print(f'Exists      : {OUT_CSV.exists()}')
print(f'FORCE_RERUN : {FORCE_RERUN}')

Output path : /Users/Sophia/NYP/src/mc_baseline_data/mc_baseline_n100_start42.csv
Exists      : True
FORCE_RERUN : True


In [ ]:
import time

if OUT_CSV.exists() and not FORCE_RERUN:
    print(f'[CACHED] {OUT_CSV.name} — set FORCE_RERUN=True to regenerate.')
else:
    if OUT_CSV.exists():
        OUT_CSV.unlink()
        print(f'Deleted old CSV: {OUT_CSV.name}')

    print(f'Running {N_SEEDS} seeds × 80yr with {N_WORKERS} workers …')
    t0 = time.time()
    run_mc_baseline(
        n_seeds=N_SEEDS,
        seed_start=SEED_START,
        n_workers=N_WORKERS,
        out_csv=str(OUT_CSV),
        progress=True,
    )
    elapsed = (time.time() - t0) / 60
    print(f'\nDone in {elapsed:.1f} min → {OUT_CSV.name}')

Deleted old CSV: mc_baseline_n100_start42.csv
Running 100 seeds × 80yr with 3 workers …
[mc_baseline] 100 baseline sims × 80yr each, 3 workers
[mc_baseline] writing → /Users/Sophia/NYP/src/mc_baseline_data/mc_baseline_n100_start42.csv
[INIT] Stable population: 1,500 established patients scheduled across 1825 warmup days.
[INIT] Stable population: 1,500 established patients scheduled across 1825 warmup days.
[INIT] Stable population: 1,500 established patients scheduled across 1825 warmup days.
  Year   5/80  |  pool  3,141  |  mortality     0  |     2.0s elapsed
  Year   5/80  |  pool  3,170  |  mortality     0  |     2.0s elapsed
  Year   5/80  |  pool  3,116  |  mortality     0  |     2.1s elapsed
  Year  10/80  |  pool  4,458  |  mortality     0  |     3.8s elapsed
  Year  10/80  |  pool  4,428  |  mortality     0  |     3.9s elapsed
  Year  10/80  |  pool  4,395  |  mortality     0  |     3.9s elapsed
  Year  15/80  |  pool  5,381  |  mortality   166  |     6.5s elapsed
  Year  15/

In [ ]:
# ── quick sanity check ────────────────────────────────────────────────────
import pandas as pd
import numpy as np

WARMUP = cfg.WARMUP_YEARS
METRICS = [
    'revenue_capture_rate_pct',
    'population_capture_rate_pct',
    'mean_wait_primary_days',
    'mean_wait_secondary_days',
    'ltfu_rate_primary_pct',
    'ltfu_rate_secondary_pct',
]

df = pd.read_csv(OUT_CSV)
records = []
for seed, sdf in df.groupby('seed'):
    row = {'seed': seed}
    for m in METRICS:
        sub = sdf[sdf['metric'] == m]
        if sub.empty:
            row[m] = np.nan
            continue
        if sub['year'].nunique() > 1:
            sub = sub[sub['year'] >= WARMUP]
        row[m] = sub['value'].mean()
    records.append(row)

summary = pd.DataFrame(records).set_index('seed')
print(f'Seeds loaded: {len(summary)}')
print(f'Metrics     : {list(summary.columns)}')
print()
summary.describe().round(3)

## Next Steps

Once this baseline CSV is ready, rerun the scenario notebooks:

1. **`capacity_scenario_boxplots.ipynb`** — delete CSVs in `src/mc_scenario_data/capacity_scenarios/`, then run all cells
2. **`capacity_scenario_integer_boxplots.ipynb`** — delete CSVs in `src/mc_scenario_data/capacity_scenarios_integer/`, then run all cells  
   *(skip if integer runs are already in progress with current parameters)*